This notebook combines GENCODE and our outlier calling to find the regions nearby genes which have at least one outlier. We can filter our genetic data by these regions, removing a lot of unnecessary annotation work. Our goal is a file of genomic coordinates to subset to, which can be input into the annotation workflow.

Similarly, we will create a file of sample IDs to subset by. This is useful because there are many samples in the genetic data which do not have expression measurements, so they can be removed.

This notebook should take about 1 minute to run.

## Setup

In [1]:
FLANK <- 1e4

In [2]:
library(data.table)

## Read and parse GENCODE file


In [3]:
g <- fread(
  'gencode.v49.annotation.gtf.gz',
  col.names=c('seqname','source','feature','start','end','score','strand','frame','attribute')
)[feature=='gene']

Warning message in fread("gencode.v49.annotation.gtf.gz", col.names = c("seqname", :
“Detected 1 column names but the data has 9 columns (i.e. invalid file). Added 8 extra default column names at the end.”


In [4]:
head(g)

seqname,source,feature,start,end,score,strand,frame,attribute
<chr>,<chr>,<chr>,<int>,<int>,<chr>,<chr>,<chr>,<chr>
chr1,HAVANA,gene,11121,24894,.,+,.,"gene_id ""ENSG00000290825.2""; gene_type ""lncRNA""; gene_name ""DDX11L16""; level 2; tag ""overlaps_pseudogene"";"
chr1,HAVANA,gene,12010,13670,.,+,.,"gene_id ""ENSG00000223972.6""; gene_type ""transcribed_unprocessed_pseudogene""; gene_name ""DDX11L1""; level 2; hgnc_id ""HGNC:37102""; havana_gene ""OTTHUMG00000000961.2"";"
chr1,HAVANA,gene,14356,30744,.,-,.,"gene_id ""ENSG00000310526.1""; gene_type ""lncRNA""; gene_name ""WASH7P""; level 2; hgnc_id ""HGNC:38034""; tag ""ncRNA_host""; tag ""overlapping_locus""; tag ""overlaps_pseudogene"";"
chr1,HAVANA,gene,14696,24886,.,-,.,"gene_id ""ENSG00000227232.6""; gene_type ""transcribed_unprocessed_pseudogene""; gene_name ""WASH7P""; level 2; hgnc_id ""HGNC:38034""; tag ""overlapping_locus""; havana_gene ""OTTHUMG00000000958.1"";"
chr1,ENSEMBL,gene,17369,17436,.,-,.,"gene_id ""ENSG00000278267.1""; gene_type ""miRNA""; gene_name ""MIR6859-1""; level 3; hgnc_id ""HGNC:50039"";"
chr1,HAVANA,gene,28589,31109,.,+,.,"gene_id ""ENSG00000243485.6""; gene_type ""lncRNA""; gene_name ""MIR1302-2HG""; level 2; hgnc_id ""HGNC:52482""; tag ""ncRNA_host""; havana_gene ""OTTHUMG00000000959.2"";"


In [5]:
g <- g[
  ][, gene_id := sub('.*"(.*)"', '\\1', tstrsplit(g$attribute,';')[[1]])
  ][, startflank := pmax(start - FLANK, 1)
  ][,   endflank :=        end + FLANK
  ][, -c('source','feature','score','frame','attribute')
]

head(g)

seqname,start,end,strand,gene_id,startflank,endflank
<chr>,<int>,<int>,<chr>,<chr>,<dbl>,<dbl>
chr1,11121,24894,+,ENSG00000290825.2,1121,34894
chr1,12010,13670,+,ENSG00000223972.6,2010,23670
chr1,14356,30744,-,ENSG00000310526.1,4356,40744
chr1,14696,24886,-,ENSG00000227232.6,4696,34886
chr1,17369,17436,-,ENSG00000278267.1,7369,27436
chr1,28589,31109,+,ENSG00000243485.6,18589,41109


In [6]:
# File created from outlier calling step
indiv_gene_z <- fread('indiv_gene_z.tsv')
head(indiv_gene_z)

indiv,gene,z
<chr>,<chr>,<dbl>
NA06985,ENSG00000002745.13,-1.20296198
NA07000,ENSG00000002745.13,-0.84021613
NA11919,ENSG00000002745.13,-0.17164761
NA11933,ENSG00000002745.13,0.05810866
NA12003,ENSG00000002745.13,0.25246033
NA12286,ENSG00000002745.13,0.66793385


In [7]:
length(unique(indiv_gene_z$gene))
length(unique(g$gene_id))

# We only care about genes present in the expression data.
g <- g[gene_id %in% indiv_gene_z$gene]
nrow(g)

[1] 3779

[1] 78691

[1] 3779

## Create region/sample filter files

In [8]:
# More specifically, we only care about SNVs within <FLANK> of the genes present in the expression data.
fwrite(file='regions_near_expression_genes.tsv', sep='\t', col.names=F,
  x = g[, .(seqname, startflank, endflank)] )

In [9]:
# We only care about samples that have expression data.
writeLines(unique(indiv_gene_z$indiv), 'samples_with_expression_data.txt')

Information about the expression genes' start/end will be useful later when calculating the nearest TSS & TES to each SNV, so we write this information as well.

In [10]:
fwrite(g, 'gencode_expression_genes_info.tsv')

### Subsetting CADD annotations
This code block is commented out because you do not need to run it; it has been pre-prepared for you as it would take a few hours to run.

The original file of CADD v1.6 annotations downloaded [here](https://cadd.gs.washington.edu/download) is +300GB which is impractical. Therefore, we subset it to only the regions near expression genes to reduce its size before input to the workflow. 

In [11]:
# tabix (unlike bcftools which is used in the annotation workflow)
# will duplicate records if you give it overlapping regions.
# Therefore we must collapse 
#collapsed_regions <- g[
#  ][, by=chr
#    , grp := cumsum(start>shift(cummax(endflank), fill=-1))
#  ][, by = .(chr,grp)
#    , .(start=min(startflank), end=max(endflank)),
#  ][, grp := NULL
#  ][, chr := sub('chr','',chr)
#] |>  fwrite('1kG_regions_near_expression_genes-collapsed.tsv', sep='\t', col.names=F)

#system('gcloud storage cp gs://fc-secure-e1fdce8e-0368-4406-8af0-d2b5cb924294/data/annotation/whole_genome_SNVs_inclAnno.tsv.gz.tbi .')
#system('gcloud storage cp gs://fc-secure-e1fdce8e-0368-4406-8af0-d2b5cb924294/data/annotation/whole_genome_SNVs_inclAnno.tsv.gz     .')

# Keep only genetic regions near gene flanks, and only the annotation columns we want
#system(paste(
#  'tabix -R "1kG_regions_near_expression_genes-collapsed.tsv whole_genome_SNVs_inclAnno.tsv.gz',
#  " | awk -v OFS='\t' '{print $1,$2,$3,$4,$11,$12,$36,$37,$38,$39,$40,$41,$42,$43,$44,$45,$46,$78,$79,$134}'",
#  ' | bgzip > 1kG_cadd_subset.tsv.bgz'
#)

#system('tabix -s1 -b2 -e3 1kG_cadd_subset.tsv.bgz')# Index the new subset CADD file

#system('gcloud storage cp 1kG_cadd_subset* $WORKSPACE_BUCKET/data/annotation/')

## Write

In [12]:
system('gcloud storage cp samples_with_expression_data.txt  $WORKSPACE_BUCKET/data/derived')
system('gcloud storage cp regions_near_expression_genes.tsv $WORKSPACE_BUCKET/data/derived')
system('gcloud storage cp gencode_expression_genes_info.tsv $WORKSPACE_BUCKET/data/derived')